# GPU benchmark for the `radiotel` FX correlator

Runs the `torch-cuda` backend through the same sweep as `benchmarks/bench_correlator.py`
(N = 2…64 antennas, 256/1024/4096 channels, 2¹⁸ samples per antenna) and produces
`correlator_gpu.csv`. Copy that file into `benchmarks/results/` and run
`python benchmarks/make_figures.py` — the GPU rows replace the *pending* placeholders.

**Before running:** *Runtime → Change runtime type → GPU* (a T4 is fine).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## Get the code
Option A clones the `revamp` branch from GitHub (it must be pushed first).
Option B: upload a zip of the repository with the file browser on the left, then unzip it into `repo/`.

In [ ]:
REPO = "https://github.com/alireza-astane/Radio_Telescope_Signal_Processing.git"
BRANCH = "revamp"
USE_ZIP = True   # True: upload repo.zip to /content first (file browser on the left); False: clone from GitHub
if USE_ZIP:
    !rm -rf /content/repo && unzip -q -o /content/repo.zip -d /content/repo
else:
    !rm -rf /content/repo && git clone --depth 1 -b $BRANCH $REPO /content/repo

In [ ]:
# Locate the checkout wherever it landed (clone, or a zip that unpacked one level deeper).
import glob, os, sys
hits = sorted(glob.glob("/content/**/src/radiotel/__init__.py", recursive=True), key=len)
assert hits, "radiotel not found under /content - rerun the 'Get the code' cell (check the zip was uploaded)"
ROOT = hits[0].split("/src/radiotel/")[0]
os.chdir(ROOT)
sys.path.insert(0, os.path.join(ROOT, "src"))
print("repository root:", ROOT)
!pip -q install -e .
!make lib

In [ ]:
import torch
from radiotel.correlator import available_backends
print(torch.__version__, torch.cuda.get_device_name(0))
print(available_backends())   # must include 'torch-cuda'


## Correctness check on the GPU before timing

In [ ]:
import numpy as np
from radiotel import correlator, signals
rng = np.random.default_rng(0)
v = signals.complex_noise(rng, (16, 1 << 14)).astype(np.complex64)
ref = correlator.correlate(v, 256, "numpy-gemm")
gpu = correlator.correlate(v, 256, "torch-cuda")
print("max relative difference:", np.max(np.abs(gpu - ref)) / np.max(np.abs(ref)))

## Run the sweep (≈ 1–2 minutes)

In [ ]:
!python benchmarks/bench_correlator.py --gpu-only

In [ ]:
import pandas as pd
df = pd.read_csv("benchmarks/results/correlator_gpu.csv")
df.pivot_table(index="n_ant", columns="nchan", values="rtf_xstep")

## Download the results

In [ ]:
from google.colab import files
files.download("benchmarks/results/correlator_gpu.csv")